# 01 — EDA and feature engineering

Exploratory pass over the seed dataset: price distributions by brand / model / era, originality penalty magnitudes, and a sanity check that the engineered features map cleanly to known vintage-guitar lore (e.g., the 1955 scalloped-bracing cliff on Gibson, the 1969 Brazilian → Indian rosewood shift on Martin).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from gibson_price.ingest.vg_price_guide import load_seed
from gibson_price.features.build import build_feature_frame

listings = load_seed(Path.cwd().parent / 'data' / 'seed' / 'gibson_acoustic_seed.csv')
df = build_feature_frame(listings)
print(f'Rows: {len(df)}')
df.head()

In [ ]:
# Price distribution by brand
df.groupby('brand')['price_usd'].describe().round(0)

In [ ]:
# Median by Gibson model x era
(df[df['brand'] == 'Gibson']
  .groupby(['model_family', 'era_segment'])['price_usd']
  .median()
  .unstack()
  .round(0))

In [ ]:
# Refinish penalty: median price within (brand, model, era) split by refinished flag
(df.groupby(['brand', 'model_family', 'era_segment', 'refinished'])['price_usd']
  .median()
  .unstack('refinished')
  .dropna()
  .assign(refin_pct=lambda x: (x[1] / x[0] - 1) * 100)
  .round(0)
  .head(20))